# Non-Python workers with custom commands

> **Runtime compatibility:** this example uses the Rust runtime. The subprocess executor selects the OpenSSL shell wrapper through its `worker_commands` configuration.

Tierkreis works easiest when running with python workers.
Still it supports using arbitrary workers that provide a binary to run.
In this example we will look at using a shell script as a worker.
Conceptually, this also works with other programming languages, for example if you want to use a legacy HPC application.

To make this work we need to introduce three changes:
1. (Optional) We should provide an interface definition using a type spec. This allows tierkreis to generate the api and use them during graph construction.
2. Rewrite or wrap the script such that it follows the tierkreis contract. In short we have to change how we read inputs and write outputs.
3. Configure the subprocess executor to use the wrapper for this worker namespace.


## Preliminaries

For this example we're going to use the auth worker which already has some predefined functionality.
Make sure to familiarize yourself with its contents.
In this example we're going to use a shell script with the following contents as a worker, which is contained in the `openssl_worker`.

```bash
#!/usr/bin/env bash
pk_file=$1
passphrase=$2
numbits=$3

openssl genrsa -out $pk_file -aes128 -passout "file:$passphrase" $numbits
openssl rsa -in $pk_file -passin "file:$passphrase" -pubout -out public-out
```

## Defining the interface
To do this we can define a TypeSpec dsl.

```
@portmapping
model Outputs {
    private_key: bytes
    public_key: bytes
}

interface openssl_worker {
    genrsa (
        numbits: int,
        passphrase: bytes
    ): Outputs 
}

```
It defines two outputs `private_key` and `public_key` consuming a `passphrase` and `numbits`.
The most generic type for such values is bytes, as we will be reading and writing from file directly.
Now we can generate the stubs using the cli `tkr init stubs` or from python.

In [ ]:
from pathlib import Path

from tierkreis.namespace import Namespace

if __name__ == "__main__":
    tsp_path = (
        Path().parent
        / "example_workers"
        / "openssl_worker"
        / "tkr_openssl_worker_impl"
        / "schema.tsp"
    )
    namespace = Namespace.from_spec_file(tsp_path)
    namespace.write_stubs(tsp_path.parent.parent / "api" / "api.py")

## Adapting the script to the tierkreis contract

Tierkreis writes intermediate values to storage and expects outputs to be written there as well. The subprocess executor passes the wrapper one argument: the path to a JSON worker-call definition. Its `inputs` and `outputs` objects map port names to the corresponding files.

The wrapper uses `jq` to extract those paths, then invokes OpenSSL. This example therefore requires Bash, `jq`, and `openssl` to be installed.

The relevant part of the adapted script is:
```bash
worker_args=${1:?Expected the Tierkreis worker call definition path}
numbits_file=$(jq -r '.inputs.numbits' "$worker_args")
passphrase_file=$(jq -r '.inputs.passphrase' "$worker_args")
private_key_file=$(jq -r '.outputs.private_key' "$worker_args")
public_key_file=$(jq -r '.outputs.public_key' "$worker_args")
numbits=$(<"$numbits_file")
openssl genrsa -out "$private_key_file" -aes128 -passout "file:$passphrase_file" "$numbits"
openssl rsa -in "$private_key_file" -passin "file:$passphrase_file" -pubout -out "$public_key_file"
```

## Building a graph using the script
Were going to build a graph that checks whether we successfully signed a message with a generated private key.


In [ ]:
from auth_worker import sign, verify
from openssl_worker import Outputs, genrsa
from tierkreis.builder import Graph
from tierkreis.models import TKR, EmptyModel


def signing_graph():
    g = Graph(EmptyModel, TKR[bool])
    message = g.const("dummymessage")
    passphrase = g.const(b"dummypassphrase")

    key_pair: Outputs = g.task(genrsa(g.const(4096), passphrase))
    private_key: TKR[bytes] = key_pair.private_key
    public_key: TKR[bytes] = key_pair.public_key

    signing_result = g.task(sign(private_key, passphrase, message)).hex_signature
    verification_result = g.task(verify(public_key, signing_result, message))
    return g.finish_with_outputs(verification_result)

## Configuring the subprocess executor

The runtime configuration uses the normal subprocess executor for installed Python workers and overrides only `openssl_worker`. The values are an argv list, not a shell command; Tierkreis appends the worker-call JSON path as the final argument.

```toml
[executors.subprocess]
type = "Subprocess"
subprocess_storage_name = "file"
output_storage_name = "file"

[executors.subprocess.worker_commands]
openssl_worker = ["example_workers/openssl_worker/tkr_openssl_worker_impl/main.sh"]
```

In [ ]:
from tierkreis.runtime import load_runtime_config, new_from_config

config = load_runtime_config(Path("runtime-signing.toml"))
runtime = await new_from_config(config)

## Running the workflow

The graph now runs entirely through the Rust runtime. The OpenSSL task uses the configured shell wrapper, while the installed auth worker continues to use its default `tkr-auth-worker` command.

In [ ]:
with runtime:
    workflow_id = await runtime.save_workflow("signing_graph", signing_graph())
    run_id = await runtime.start_new_run(workflow_id, {})
    await runtime.wait_for(run_id)
    is_verified = await runtime.get_outputs(run_id)

print(is_verified)